# A Prism as a Fourier Transform for Light

White light is a superposition of many frequencies (colors), just like a complex signal is a sum of sinusoids. A **prism** decomposes that mixture into its constituent frequency components — each wavelength bends at a different angle due to **dispersion** (the refractive index $n(\lambda)$ depends on wavelength), spatially separating them. This is exactly what the Fourier transform does.

| Fourier Transform | Prism |
|---|---|
| Input signal $f(t)$ | White light (superposition of wavelengths) |
| Output spectrum $\hat{f}(\omega)$ | Rainbow (each color separated spatially) |
| Decomposes into sinusoidal modes $e^{i\omega t}$ | Decomposes into monochromatic waves (pure colors) |
| Frequency axis $\omega$ | Angular position on a screen |

Both are **linear operations**: $T[\alpha f + \beta g] = \alpha\,T[f] + \beta\,T[g]$.

### Important caveat: only the power spectrum

A prism spatially separates frequencies, but any detector (eye, CCD, film) measures **intensity**:

$$
I(\lambda) = |E(\lambda)|^2 = A(\lambda)^2
$$

The complex field $E(\lambda) = A(\lambda)\,e^{i\phi(\lambda)}$ has both amplitude and phase, but squaring discards the phase. So a prism really gives you the **power spectrum** $|G(k)|^2$, not the full complex Fourier transform.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, interact


def plot_prism_dispersion():
    """Visualize how a prism separates white light into its frequency components."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left panel: prism schematic ---
    ax = axes[0]
    prism = plt.Polygon(
        [[2, 0.5], [4, 3.5], [6, 0.5]],
        closed=True,
        facecolor="lightskyblue",
        edgecolor="steelblue",
        linewidth=2,
        alpha=0.5,
    )
    ax.add_patch(prism)

    # White input beam
    ax.annotate(
        "",
        xy=(2.8, 2.2),
        xytext=(0, 2.2),
        arrowprops=dict(arrowstyle="->, head_width=0.2", color="gray", lw=3),
    )
    ax.text(0.2, 2.5, "White light", fontsize=11, color="gray", fontweight="bold")

    # Dispersed output beams (rainbow)
    colors = ["#FF0000", "#FF7F00", "#FFFF00", "#00FF00", "#0000FF", "#4B0082", "#8B00FF"]
    labels = ["Red", "", "", "Green", "", "", "Violet"]
    base_angle = -0.25
    for i, (c, lbl) in enumerate(zip(colors, labels)):
        angle = base_angle - i * 0.08
        end_x = 8.5
        end_y = 1.8 + angle * (end_x - 5.0)
        ax.annotate(
            "",
            xy=(end_x, end_y),
            xytext=(5.0, 1.8),
            arrowprops=dict(arrowstyle="->, head_width=0.12", color=c, lw=2),
        )
        if lbl:
            ax.text(end_x + 0.1, end_y, lbl, fontsize=9, color=c, va="center")

    ax.set_xlim(-0.5, 10)
    ax.set_ylim(-0.5, 4.5)
    ax.set_aspect("equal")
    ax.set_title("Prism: Dispersion Separates Wavelengths", fontsize=12)
    ax.axis("off")

    # --- Right panel: power spectrum (what a detector sees) ---
    ax2 = axes[1]
    wavelengths = np.linspace(380, 750, 500)

    # Simulate a "white light" spectrum with a couple of spectral features
    spectrum = np.ones_like(wavelengths)
    spectrum += 0.4 * np.exp(-((wavelengths - 486) ** 2) / (2 * 15**2))
    spectrum += 0.3 * np.exp(-((wavelengths - 656) ** 2) / (2 * 15**2))

    def wavelength_to_rgb(wl):
        """Approximate visible wavelength (nm) to RGB."""
        if 380 <= wl < 440:
            r, g, b = -(wl - 440) / (440 - 380), 0.0, 1.0
        elif 440 <= wl < 490:
            r, g, b = 0.0, (wl - 440) / (490 - 440), 1.0
        elif 490 <= wl < 510:
            r, g, b = 0.0, 1.0, -(wl - 510) / (510 - 490)
        elif 510 <= wl < 580:
            r, g, b = (wl - 510) / (580 - 510), 1.0, 0.0
        elif 580 <= wl < 645:
            r, g, b = 1.0, -(wl - 645) / (645 - 580), 0.0
        elif 645 <= wl <= 750:
            r, g, b = 1.0, 0.0, 0.0
        else:
            r, g, b = 0.0, 0.0, 0.0
        return (r, g, b)

    colors_rgb = [wavelength_to_rgb(wl) for wl in wavelengths]
    for i in range(len(wavelengths) - 1):
        ax2.fill_between(wavelengths[i : i + 2], 0, spectrum[i : i + 2], color=colors_rgb[i])

    ax2.set_title(r"Detector sees $|E(\lambda)|^2$: power spectrum only", fontsize=12)
    ax2.set_xlabel("Wavelength (nm)")
    ax2.set_ylabel("Intensity")
    ax2.set_xlim(380, 750)
    ax2.set_ylim(0)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


plot_prism_dispersion()

## Diffraction Grating: A Closer Analog to the Fourier Transform

A **diffraction grating** is a surface with many equally spaced parallel slits (typically hundreds to thousands per mm). When light hits the grating, each slit acts as a new wave source. These waves **interfere**:

- **Constructive interference** (bright) when the path difference is a whole number of wavelengths:

$$
d \sin\theta = m\lambda, \quad m = 0, \pm 1, \pm 2, \ldots
$$

where $d$ is the slit spacing, $\theta$ is the diffraction angle, $\lambda$ is wavelength, and $m$ is the order.

- **Destructive interference** (dark) at other angles.

### Why it's a true Fourier transform

If the grating has aperture transmission $f(x)$, the **Fraunhofer diffraction** pattern is:

$$
E(\theta) \propto \int f(x)\, e^{-ikx\sin\theta}\, dx
$$

This is literally a Fourier transform of $f(x)$! The grating's physics **computes** the FT through interference — unlike a prism which uses refraction.

### Phase information

Both prism and grating lose phase at the detector (we measure $|E|^2$). To recover phase, you need **interferometric techniques** like holography (mixing with a reference beam).

In [ ]:
def plot_diffraction_grating(N_slits=5, d=5.0):
    """
    Visualize diffraction grating: slit geometry and the resulting
    far-field intensity pattern (|FT|^2 of the aperture function).
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Left: grating schematic ---
    ax = axes[0]
    slit_width = 0.3
    slit_positions = np.arange(N_slits) * d - (N_slits - 1) * d / 2

    for i, pos in enumerate(slit_positions):
        if i == 0:
            ax.fill_between(
                [pos - d / 2, pos - slit_width / 2], -1, 1, color="dimgray", alpha=0.8
            )
        ax.fill_between(
            [pos + slit_width / 2, pos + d / 2 if i < N_slits - 1 else pos + d / 2],
            -1,
            1,
            color="dimgray",
            alpha=0.8,
        )
        ax.fill_between(
            [pos - slit_width / 2, pos + slit_width / 2],
            -1,
            1,
            color="lightyellow",
            alpha=0.6,
        )
        for angle in np.linspace(-0.6, 0.6, 5):
            ax.annotate(
                "",
                xy=(pos + 2 * np.sin(angle), 1 + 2 * np.cos(angle)),
                xytext=(pos, 1),
                arrowprops=dict(arrowstyle="-", color="dodgerblue", lw=0.8, alpha=0.4),
            )

    if N_slits >= 2:
        ax.annotate(
            "",
            xy=(slit_positions[1], -1.3),
            xytext=(slit_positions[0], -1.3),
            arrowprops=dict(arrowstyle="<->", color="red", lw=1.5),
        )
        ax.text(
            (slit_positions[0] + slit_positions[1]) / 2,
            -1.6,
            f"d = {d}",
            ha="center",
            fontsize=10,
            color="red",
        )

    ax.set_title(f"Diffraction Grating ({int(N_slits)} slits)", fontsize=12)
    ax.set_xlim(slit_positions[0] - d, slit_positions[-1] + d)
    ax.set_ylim(-2, 4)
    ax.axis("off")

    # --- Right: far-field diffraction pattern = |FT of aperture|^2 ---
    ax2 = axes[1]

    x_fine = np.linspace(-N_slits * d, N_slits * d, 10000)
    aperture = np.zeros_like(x_fine)
    for pos in slit_positions:
        aperture += (
            (x_fine >= pos - slit_width / 2) & (x_fine <= pos + slit_width / 2)
        ).astype(float)

    dx = x_fine[1] - x_fine[0]
    ft = np.fft.fftshift(np.fft.fft(aperture)) * dx
    freqs = np.fft.fftshift(np.fft.fftfreq(len(x_fine), d=dx)) * 2 * np.pi

    intensity = np.abs(ft) ** 2
    intensity /= intensity.max()

    mask = np.abs(freqs) < 8
    ax2.plot(freqs[mask], intensity[mask], color="navy", linewidth=2)
    ax2.fill_between(freqs[mask], intensity[mask], alpha=0.15, color="navy")

    for m in range(-3, 4):
        k_peak = 2 * np.pi * m / d
        if np.abs(k_peak) < 8:
            ax2.axvline(k_peak, color="red", linestyle="--", alpha=0.4)
            ax2.text(k_peak, 1.05, f"m={m}", ha="center", fontsize=8, color="red")

    ax2.set_title(
        r"Far-field intensity $|E(k)|^2 \propto |\mathcal{F}\{f(x)\}|^2$", fontsize=12
    )
    ax2.set_xlabel("Spatial frequency k")
    ax2.set_ylabel("Normalized intensity")
    ax2.set_ylim(0, 1.15)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


interact(
    plot_diffraction_grating,
    N_slits=FloatSlider(value=5, min=2, max=20, step=1, description="N slits"),
    d=FloatSlider(value=5.0, min=2.0, max=10.0, step=0.5, description="d (spacing)"),
);

## Prism vs. Diffraction Grating — Summary

| | Prism | Diffraction Grating |
|---|---|---|
| **Mechanism** | Refraction (wavelength-dependent $n(\lambda)$) | Interference between slit sources |
| **Relation to FT** | Analogy only | Literally computes $\int f(x)\,e^{-ikx}\,dx$ |
| **Resolution** | Limited by material dispersion | Improves with number of slits $N$ |
| **What you detect** | $|E(\lambda)|^2$ (power spectrum) | $|E(k)|^2$ (power spectrum) |
| **Phase recovery** | Requires interferometry | Requires interferometry (e.g., holography) |

**Key takeaway:** Both devices spatially separate frequencies, acting as physical Fourier transforms. But what we *observe* is always $|G(k)|^2$ — the phase is lost at the detector. The grating is the more faithful analog because its underlying physics (interference) is the actual mechanism of the Fourier integral.